## Import Libraries

In [1]:
import pandas as pd
import os
import numpy as np

## verify everything before starting

In [2]:
# Run this to verify everything before starting Stage 1
import os, pandas as pd

checks = {
    'cms beneficiary 2010': '../data/raw/cms/DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv',
    'cms inpatient':        '../data/raw/cms/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv',
    'cms outpatient':       '../data/raw/cms/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv',
    'synthea patients':     '../data/raw/synthea/patients.csv',
    'synthea conditions':   '../data/raw/synthea/conditions.csv',
    'synthea encounters':   '../data/raw/synthea/encounters.csv',
    'insurance':            '../data/raw/insurance.csv',
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {name}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    ins = pd.read_csv('../data/raw/insurance.csv')
    assert len(ins) == 1338, f"insurance.csv has {len(ins)} rows, expected 1338"
    print("\n✅ Stage 0 complete — all data files present. Proceed to Stage 1.")
else:
    print("\n❌ Missing files above — complete downloads before proceeding.")

  ✅  cms beneficiary 2010: ../data/raw/cms/DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv
  ✅  cms inpatient: ../data/raw/cms/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
  ✅  cms outpatient: ../data/raw/cms/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv
  ✅  synthea patients: ../data/raw/synthea/patients.csv
  ✅  synthea conditions: ../data/raw/synthea/conditions.csv
  ✅  synthea encounters: ../data/raw/synthea/encounters.csv
  ✅  insurance: ../data/raw/insurance.csv

✅ Stage 0 complete — all data files present. Proceed to Stage 1.


## Inspect the Synthea 5k output

Before generating the spine, verify Synthea produced usable data:

In [3]:
import pandas as pd

from pathlib import Path
data_dir = Path().resolve().parent / 'data' / 'raw' / 'synthea'

patients  = pd.read_csv(data_dir / 'patients.csv')
conditions = pd.read_csv(data_dir / 'conditions.csv')
encounters = pd.read_csv(data_dir / 'encounters.csv')

print(f"Synthea patients:   {len(patients):,}")
print(f"Synthea conditions: {len(conditions):,}")
print(f"Synthea encounters: {len(encounters):,}")

# Filter to working age
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])
patients['age'] = (pd.Timestamp('today') - patients['BIRTHDATE']).dt.days / 365.25
patients_wa = patients[(patients['age'] >= 25) & (patients['age'] <= 65)]
print(f"Working-age patients (25–65): {len(patients_wa):,}")

# Check condition descriptions are present
print("\nSample condition descriptions:")
print(conditions['DESCRIPTION'].value_counts().head(10))

Synthea patients:   5,709
Synthea conditions: 211,882
Synthea encounters: 350,677
Working-age patients (25–65): 2,852

Sample condition descriptions:
DESCRIPTION
Medication review due (situation)    42183
Stress (finding)                     16740
Gingivitis (disorder)                15604
Full-time employment (finding)       15414
Part-time employment (finding)        9677
Limited social contact (finding)      5976
Social isolation (finding)            5923
Viral sinusitis (disorder)            5817
Not in labor force (finding)          5296
Gingival disease (disorder)           4515
Name: count, dtype: int64


 ## Extract condition prevalence rates from Synthea

This is the key thing we need from Synthea — how common are MSK, metabolic, and MH conditions in a working-age population. These rates drive the synthetic spine's condition flags

In [4]:
import pandas as pd
import numpy as np

from pathlib import Path
data_dir = Path().resolve().parent / 'data' / 'raw' / 'synthea'

patients  = pd.read_csv(data_dir / 'patients.csv')
conditions = pd.read_csv(data_dir / 'conditions.csv')
encounters = pd.read_csv(data_dir / 'encounters.csv')

# Working-age filter
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])
patients['age'] = (pd.Timestamp('today') - patients['BIRTHDATE']).dt.days / 365.25
patients_wa = patients[(patients['age'] >= 25) & (patients['age'] <= 65)].copy()

# Tag condition clusters by description text
msk_keywords      = r'back pain|joint pain|fibromyalgia|spondyl|osteoarthr|rotator cuff|disc disorder|lumbar|cervical'
metabolic_keywords = r'diabetes|obesity|hyperlipid|metabolic syndrome|prediabetes'
mh_keywords       = r'major depression|depressive disorder|anxiety|panic disorder|adjustment disorder|PTSD|post.traumatic'

conditions['cluster'] = None
conditions.loc[conditions['DESCRIPTION'].str.contains(msk_keywords, case=False, na=False), 'cluster'] = 'MSK'
conditions.loc[conditions['DESCRIPTION'].str.contains(metabolic_keywords, case=False, na=False), 'cluster'] = 'Metabolic'
conditions.loc[conditions['DESCRIPTION'].str.contains(mh_keywords, case=False, na=False), 'cluster'] = 'MH'

# Working-age flagged conditions only
flagged = conditions[
    conditions['cluster'].notna() &
    conditions['PATIENT'].isin(patients_wa['Id'])
]
print("Flagged conditions by cluster:")
print(flagged['cluster'].value_counts())

# Per-patient flags (binary: has / doesn't have each cluster)
patient_flags = (
    flagged.groupby(['PATIENT', 'cluster'])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
    .reset_index()
)
# Ensure all three columns exist
for col in ['MSK', 'Metabolic', 'MH']:
    if col not in patient_flags.columns:
        patient_flags[col] = 0

patient_flags = patient_flags.rename(columns={
    'MSK': 'has_msk_flag',
    'Metabolic': 'has_metabolic_flag',
    'MH': 'has_mh_flag'
})

# Fill patients with no flagged conditions
all_wa = patients_wa[['Id']].copy()
patient_flags_full = all_wa.merge(patient_flags, left_on='Id', right_on='PATIENT', how='left').fillna(0)

# Extract prevalence rates
msk_rate      = patient_flags_full['has_msk_flag'].mean()
metabolic_rate = patient_flags_full['has_metabolic_flag'].mean()
mh_rate       = patient_flags_full['has_mh_flag'].mean()

print(f"\nSynthea prevalence rates (working-age):")
print(f"  MSK:       {msk_rate:.1%}")
print(f"  Metabolic: {metabolic_rate:.1%}")
print(f"  MH:        {mh_rate:.1%}")

# Save for use in Step 5
prevalence = {'msk': msk_rate, 'metabolic': metabolic_rate, 'mh': mh_rate}

Flagged conditions by cluster:
cluster
Metabolic    4277
MSK          1049
MH            359
Name: count, dtype: int64

Synthea prevalence rates (working-age):
  MSK:       31.3%
  Metabolic: 74.2%
  MH:        11.4%


## Extract encounter distributions from Synthea

In [5]:
# Classify encounter types
ENCOUNTER_MAP = {
    'ambulatory': 'gp_visit',
    'wellness':   'gp_visit',
    'outpatient': 'specialist_visit',
    'urgentcare': 'specialist_visit',
    'emergency':  'ed_visit',
    'inpatient':  'inpatient',
}

encounters_wa = encounters[encounters['PATIENT'].isin(patients_wa['Id'])].copy()
encounters_wa['claim_type'] = encounters_wa['ENCOUNTERCLASS'].map(ENCOUNTER_MAP).fillna('other')

enc_counts = (
    encounters_wa
    .groupby(['PATIENT', 'claim_type'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['gp_visit', 'specialist_visit', 'ed_visit']:
    if col not in enc_counts.columns:
        enc_counts[col] = 0

# Scale to approximate 6-month counts
# Synthea generates lifetime records — divide by rough lifetime visits to get 6m rate
enc_counts['gp_visit_6m']         = enc_counts['gp_visit'].apply(lambda x: min(x // 10, 12))
enc_counts['specialist_visit_6m'] = enc_counts['specialist_visit'].apply(lambda x: min(x // 15, 8))
enc_counts['ed_visit_6m']         = enc_counts['ed_visit'].apply(lambda x: min(x // 20, 4))

print("Encounter distribution (6m scaled):")
print(enc_counts[['gp_visit_6m','specialist_visit_6m','ed_visit_6m']].describe())

Encounter distribution (6m scaled):
claim_type  gp_visit_6m  specialist_visit_6m  ed_visit_6m
count       2852.000000          2852.000000  2852.000000
mean           2.874825             0.303296     0.007714
std            2.137365             0.882157     0.105663
min            0.000000             0.000000     0.000000
25%            2.000000             0.000000     0.000000
50%            2.000000             0.000000     0.000000
75%            4.000000             0.000000     0.000000
max           12.000000             8.000000     2.000000


## Generate the 50k synthetic member spine

cd allied-health-nudge
activate venv -  C:\venvs\allied-health-nudge\Scripts\Activate.ps1  
python src/synthesise.py

## Apply Synthea prevalence rates to the 50k spine

In [6]:
from pathlib import Path

data_dir = Path().resolve().parent / 'data' / 'raw'
members = pd.read_csv(data_dir / 'members.csv')

N = len(members)  # 50,000
print(f"Number of Members: {N}")

np.random.seed(42)
age_norm = (members['age'] - 25) / 40  # 0 at age 25, 1 at age 65

# Apply condition flags with age adjustment
# Older members are more likely to have MSK and metabolic conditions
members['has_msk_flag'] = (
    np.random.random(N) < (prevalence['msk'] * (0.7 + 0.6 * age_norm))
).astype(int)
members['has_metabolic_flag'] = (
    np.random.random(N) < (prevalence['metabolic'] * (0.6 + 0.8 * age_norm))
).astype(int)
members['has_mh_flag'] = (
    np.random.random(N) < prevalence['mh']  # MH is roughly flat across age
).astype(int)

members['comorbidity_count'] = members[['has_msk_flag','has_metabolic_flag','has_mh_flag']].sum(axis=1)

def assign_cluster(row):
    active = [name for name, val in [
        ('MSK', row['has_msk_flag']),
        ('Metabolic', row['has_metabolic_flag']),
        ('MH', row['has_mh_flag'])
    ] if val == 1]
    if len(active) == 0: return 'Healthy'
    if len(active) > 1:  return 'Mixed'
    return active[0]

members['condition_cluster'] = members.apply(assign_cluster, axis=1)

members.to_csv(data_dir / 'members.csv', index=False)

print("Condition cluster distribution (50k spine):")
print(members['condition_cluster'].value_counts())
print(f"\nComorbidity counts:")
print(members['comorbidity_count'].value_counts().sort_index())

Number of Members: 50000


Condition cluster distribution (50k spine):
condition_cluster
Metabolic    21356
Mixed        14051
Healthy       9580
MSK           3818
MH            1195
Name: count, dtype: int64

Comorbidity counts:
comorbidity_count
0     9580
1    26369
2    12854
3     1197
Name: count, dtype: int64


## Sample Synthea encounter profiles onto 50k members

In [7]:
def sample_visit_counts(members_df, enc_dist, n):
    sampled = enc_dist.sample(n=n, replace=True).reset_index(drop=True)
    members_df = members_df.copy()
    members_df['gp_visits_6m']         = sampled['gp_visit_6m'].values
    members_df['specialist_visits_6m'] = sampled['specialist_visit_6m'].values
    members_df['ed_visits_6m']         = sampled['ed_visit_6m'].values
    return members_df

members = sample_visit_counts(members, enc_counts, N)

## Step 8 — Generate benefits entitlement table

In [8]:
BENEFIT_ENTITLEMENTS = {
    'Bronze': {'physio': 6,  'chiro': 4,  'dietetics': 4,  'psychology': 6},
    'Silver': {'physio': 10, 'chiro': 6,  'dietetics': 6,  'psychology': 10},
    'Gold':   {'physio': 15, 'chiro': 10, 'dietetics': 10, 'psychology': 15},
}

benefit_rows = []
for _, member in members.iterrows():
    entitlements = BENEFIT_ENTITLEMENTS[member['plan_type']]
    for benefit_type, sessions_entitled in entitlements.items():
        util_prob = np.random.random()
        if util_prob < 0.45:
            sessions_used = 0
        elif util_prob < 0.75:
            sessions_used = np.random.randint(1, max(2, int(sessions_entitled * 0.4) + 1))
        elif util_prob < 0.90:
            sessions_used = np.random.randint(
                int(sessions_entitled * 0.4), int(sessions_entitled * 0.8) + 1)
        else:
            sessions_used = sessions_entitled
        benefit_rows.append({
            'member_id':         member['member_id'],
            'benefit_type':      benefit_type,
            'sessions_entitled': sessions_entitled,
            'sessions_used':     sessions_used,
            'sessions_remaining': sessions_entitled - sessions_used,
        })

benefits = pd.DataFrame(benefit_rows)
benefits.to_csv(data_dir / 'benefits.csv', index=False)
print(f"Benefits table: {len(benefits):,} rows  ({len(benefits)//4:,} members × 4 benefit types)")

Benefits table: 200,000 rows  (50,000 members × 4 benefit types)


## Build claims table and add cost estimates

In [9]:
# Allied health claims from benefits table
allied_totals = (
    benefits.groupby('member_id')['sessions_used']
    .sum()
    .reset_index()
    .rename(columns={'sessions_used': 'allied_health_claims_6m'})
)
allied_totals['allied_health_claims_6m'] = (
    allied_totals['allied_health_claims_6m'] * 0.5
).round().astype(int)  # benefits are annual → ~50% in 6m window

members = members.merge(allied_totals, on='member_id', how='left')
members['allied_health_claims_6m'] = members['allied_health_claims_6m'].fillna(0).astype(int)

members['days_since_last_allied'] = np.where(
    members['allied_health_claims_6m'] == 0,
    999,
    np.random.randint(1, 181, size=N)
)

# Cost calibration from insurance.csv
insurance = pd.read_csv(data_dir / 'insurance.csv')
insurance['age_band'] = pd.cut(insurance['age'], bins=[18,34,44,54,65], labels=['25-34','35-44','45-54','55-65'])
cost_stats = insurance.groupby(['age_band','smoker'])['charges'].agg(['mean','std']).reset_index()

def estimate_spend(row):
    band = pd.cut([row['age']], bins=[18,34,44,54,65], labels=['25-34','35-44','45-54','55-65'])[0]
    smoker = 'yes' if (row.get('has_metabolic_flag', 0) == 1 and np.random.random() < 0.3) else 'no'
    match = cost_stats[(cost_stats['age_band'] == band) & (cost_stats['smoker'] == smoker)]
    if len(match) == 0:
        return np.random.uniform(3000, 9000)
    mu, sigma = match['mean'].values[0], match['std'].values[0]
    return max(500, np.random.normal(mu * 0.5, sigma * 0.3))

members['total_spend_6m']  = members.apply(estimate_spend, axis=1).round(2)
members['total_claims_6m'] = (
    members['gp_visits_6m'] + members['specialist_visits_6m'] + members['allied_health_claims_6m']
)

members.to_csv(data_dir / 'members.csv', index=False)

claims = members[['member_id','gp_visits_6m','specialist_visits_6m','ed_visits_6m',
                   'allied_health_claims_6m','days_since_last_allied','total_claims_6m','total_spend_6m']]
claims.to_csv(data_dir / 'claims.csv', index=False)
print(f"Claims table saved: {len(claims):,} rows")

insurance.to_csv(data_dir / 'cost_reference.csv', index=False)
print(f"Cost reference saved: {len(insurance):,} rows")

Claims table saved: 50,000 rows
Cost reference saved: 1,338 rows


## Step 10 — Simulate acute events (label DGP)

Must run **after Step 7 (condition flags) and Step 8 (benefits)**.
DGP lives in `src/acute_events.py` — two causal layers:
- **Background risk**: `condition_cluster + age + comorbidity_count` (no utilisation)
- **Protective moderator**: `overall_utilisation_rate` reduces background risk up to 70%

Nudge features (`zero_allied_health_flag`, `sessions_remaining_*`, `high_gp_low_allied`)
now have genuine indirect causal paths to the label through utilisation.

In [10]:
import sys
import pandas as pd
from pathlib import Path

src_dir = Path().resolve().parent / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from acute_events import simulate_acute_events

# Read the final saved versions (condition flags + benefits from steps above)
data_dir_ae    = Path().resolve().parent / 'data' / 'raw'
members_final  = pd.read_csv(data_dir_ae / 'members.csv')
benefits_final = pd.read_csv(data_dir_ae / 'benefits.csv')

acute = simulate_acute_events(members_final, benefits_final, seed=42)
acute.to_csv(data_dir_ae / 'acute_events.csv', index=False)

pos_count = int(acute['acute_event'].sum())
neg_count = len(acute) - pos_count
spw       = neg_count / pos_count

print(f'Positive class rate : {acute["acute_event"].mean():.1%}  ({pos_count:,} events)')
print(f'scale_pos_weight    : {spw:.2f}  -- carry to Stage 4')

print('\nAcute rate by condition cluster:')
print(
    acute.groupby('condition_cluster')['acute_event']
    .agg(['mean','sum','count'])
    .rename(columns={'mean':'rate','sum':'events','count':'members'})
    .sort_values('rate', ascending=False).round(3)
)

print('\nAcute rate by utilisation quartile (protection check):')
acute['util_q'] = pd.qcut(
    acute['overall_utilisation_rate'], q=4,
    labels=['Q1 (lowest)','Q2','Q3','Q4 (highest)']
)
util_rates = acute.groupby('util_q', observed=True)['acute_event'].mean().round(3)
print(util_rates)
print(f'  Risk ratio Q1/Q4: {util_rates.iloc[0]/util_rates.iloc[-1]:.2f}x')

print('\nAcute rate by comorbidity count:')
print(acute.groupby('comorbidity_count')['acute_event'].mean().sort_index().round(3))

Positive class rate : 17.6%  (8,816 events)
scale_pos_weight    : 4.67  -- carry to Stage 4

Acute rate by condition cluster:
                    rate  events  members
condition_cluster                        
Mixed              0.353    4963    14051
MSK                0.154     589     3818
Metabolic          0.135    2874    21356
MH                 0.114     136     1195
Healthy            0.027     254     9580

Acute rate by utilisation quartile (protection check):
util_q
Q1 (lowest)     0.227
Q2              0.189
Q3              0.158
Q4 (highest)    0.121
Name: acute_event, dtype: float64
  Risk ratio Q1/Q4: 1.88x

Acute rate by comorbidity count:
comorbidity_count
0    0.027
1    0.136
2    0.345
3    0.444
Name: acute_event, dtype: float64


In [11]:
# Step 10 validation
loaded_ae = pd.read_csv(data_dir_ae / 'acute_events.csv')

assert len(loaded_ae) == 50_000,                          'acute_events row count wrong'
assert loaded_ae['member_id'].nunique() == 50_000,        'Duplicate member_ids'
assert loaded_ae['acute_event'].isin([0,1]).all(),        'Non-binary values in acute_event'
assert 0.15 <= loaded_ae['acute_event'].mean() <= 0.30,   \
    f"Positive rate {loaded_ae['acute_event'].mean():.1%} outside [15%, 30%]"

q1_r = loaded_ae.nsmallest(12_500,'overall_utilisation_rate')['acute_event'].mean()
q4_r = loaded_ae.nlargest(12_500,'overall_utilisation_rate')['acute_event'].mean()
assert q1_r > q4_r, f'Protective effect absent: Q1={q1_r:.3f}, Q4={q4_r:.3f}'

cr = loaded_ae.groupby('condition_cluster')['acute_event'].mean()
assert cr['Mixed'] > cr['Healthy'], 'Mixed should exceed Healthy'
assert cr['Mixed'] > cr['MSK'],     'Mixed should exceed MSK'

print('Step 10 validation passed')
print(f"  Positive rate: {loaded_ae['acute_event'].mean():.1%}")
print(f'  Protection: Q1={q1_r:.1%} -> Q4={q4_r:.1%} ({q1_r/q4_r:.2f}x ratio)')

Step 10 validation passed
  Positive rate: 17.6%
  Protection: Q1=22.8% -> Q4=12.1% (1.88x ratio)


## Validation Checks

In [12]:
from pathlib import Path

data_dir = Path().resolve().parent / 'data' / 'raw'
members = pd.read_csv(data_dir / 'members.csv')
benefits = pd.read_csv(data_dir / 'benefits.csv')
claims = pd.read_csv(data_dir / 'claims.csv')

assert len(members) == 50_000,                                    "Member count wrong"
assert members['member_id'].nunique() == 50_000,                  "Duplicate member_ids"
assert members['age'].between(25, 65).all(),                      "Age out of range"
assert 39 < members['age'].mean() < 46,                           "Age mean unexpected"
assert members['has_msk_flag'].mean() > 0.15,                     "Too few MSK flags"
assert members[['has_msk_flag','has_metabolic_flag','has_mh_flag',
                 'comorbidity_count','condition_cluster']].isnull().sum().sum() == 0, \
    "Null condition values in members.csv"
assert set(members['condition_cluster'].unique()) <= {'Healthy','MSK','Metabolic','MH','Mixed'}, \
    f"Unexpected cluster values: {set(members['condition_cluster'].unique())}"
assert len(benefits) == 200_000,                                   "Benefits rows wrong (50k × 4 types)"
assert (benefits['sessions_used'] <= benefits['sessions_entitled']).all(), "sessions_used exceeds entitled"
assert (benefits['sessions_remaining'] >= 0).all(),               "Negative sessions remaining"
assert (claims['allied_health_claims_6m'] == 0).mean() > 0.05, "Suspiciously few zero allied health claimants"
assert claims['allied_health_claims_6m'].mean() < 10, "Allied health claim counts unrealistically high"
assert claims['total_spend_6m'].between(0, 100_000).all(),        "Spend values out of range"

print("✅ All Stage 1 validation checks passed")
print(f"Condition clusters:\n{members['condition_cluster'].value_counts()}")

✅ All Stage 1 validation checks passed
Condition clusters:
condition_cluster
Metabolic    21356
Mixed        14051
Healthy       9580
MSK           3818
MH            1195
Name: count, dtype: int64
